# 02 - Limpeza da base integrada

## Critérios de sinalização de preços suspeitos

Os critérios abaixo identificam observações para revisão; não comprovam erro e não removem registros automaticamente.

- **Critério histórico de 3 desvios-padrão:** compara cada preço com os preços do mesmo combustível e semana em todos os municípios. Sinaliza quando há pelo menos 3 observações e o desvio absoluto em relação à média do grupo excede 3 desvios-padrão amostrais.
- **IQR local:** compara observações do mesmo município, combustível e semana. Exige pelo menos 4 observações e IQR positivo; sinaliza valores abaixo de Q1 - 1,5 × IQR ou acima de Q3 + 1,5 × IQR.
- **Flag final:** união dos dois critérios. Uma observação sinalizada por ambos aparece uma única vez no conjunto de suspeitos.

Os limites são ferramentas de triagem, não limites físicos ou comerciais de preço. Diferenças legítimas entre postos podem ser sinalizadas, especialmente em grupos pequenos ou heterogêneos. Por isso, todos os preços positivos e interpretáveis permanecem na base analítica, inclusive os sinalizados. Uma exclusão só deve ocorrer após confirmação de erro na fonte, com motivo registrado na auditoria; casos não confirmados permanecem e serão avaliados também na análise de sensibilidade do Notebook 03.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

RAIZ = Path.cwd()
if not (RAIZ / "Dados").exists() and (RAIZ.parent / "Dados").exists():
    RAIZ = RAIZ.parent
assert (RAIZ / "Dados").exists(), "Confira a pasta aberta no VS Code."

ENTRADA = RAIZ / "Dados/Limpo/precos_rmjp_ingestao.parquet"
PASTA_TABELAS = RAIZ / "tabelas"
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)

# Preserva com nome de legado o relatório da versão que removia outliers.
relatorio_antigo = PASTA_TABELAS / "outliers_removidos.csv"
relatorio_legado = PASTA_TABELAS / "outliers_removidos_legado.csv"
if relatorio_antigo.exists() and not relatorio_legado.exists():
    relatorio_antigo.replace(relatorio_legado)

print("Python:", sys.executable)
print("CWD:", Path.cwd())
print("Projeto:", RAIZ)
print("Arquivo existe:", ENTRADA.exists())

try:
    import pyarrow
    print("pyarrow OK:", pyarrow.__version__)
except Exception:
    print("ERRO: pyarrow não está instalado no ambiente ativo.")
    raise

df = pd.read_parquet(ENTRADA)
print("Registros:", len(df))
print(df.dtypes)
display(df.head())
display(df.isna().sum())

Python: c:\Users\super\Desktop\Leo\faculdade\Tcc analise combustivel\projeto\TCC_Combustiveis_revisado_20261014\.venv-1\Scripts\python.exe
CWD: c:\Users\super\Desktop\Leo\faculdade\Tcc analise combustivel\projeto\TCC_Combustiveis_revisado_20261014\notebooks
Projeto: c:\Users\super\Desktop\Leo\faculdade\Tcc analise combustivel\projeto\TCC_Combustiveis_revisado_20261014
Arquivo existe: True
pyarrow OK: 25.0.1
Registros: 69506
estado                       str
municipio                    str
revenda                      str
cnpj_original             string
logradouro                   str
numero                       str
complemento                  str
bairro                       str
cep                       string
produto                      str
data_original                str
valor_original               str
bandeira                     str
arquivo_origem               str
cnpj_revenda              string
data              datetime64[us]
valor                    Float64
dtype: obje

,estado,municipio,revenda,cnpj_original,logradouro,numero,complemento,bairro,cep,produto,data_original,valor_original,bandeira,arquivo_origem,cnpj_revenda,data,valor
0,PB,BAYEUX,FRANÇA COMBUSTÍVEIS LTDA.,09.359.690/0001-44,RUA FRANCISCO MARQUES DA FONSECA,340,NaN,IMACULADA,58309990,GASOLINA,05/01/2011,"2,54",BRANCA,ca-2011-01.csv,09359690000144,2011-01-05,2.54
1,PB,BAYEUX,FRANÇA COMBUSTÍVEIS LTDA.,09.359.690/0001-44,RUA FRANCISCO MARQUES DA FONSECA,340,NaN,IMACULADA,58309990,ETANOL,05/01/2011,"1,94",BRANCA,ca-2011-01.csv,09359690000144,2011-01-05,1.94
2,PB,BAYEUX,MOTOGAS INDUSTRIA DE COMPRESSAO E COMERCIO DE ...,01.420.327/0001-85,LOC BR 230 KM 31,S/N,NaN,ESTRADA,58309602,GASOLINA,05/01/2011,"2,48",BRANCA,ca-2011-01.csv,01420327000185,2011-01-05,2.48
3,PB,BAYEUX,MOTOGAS INDUSTRIA DE COMPRESSAO E COMERCIO DE ...,01.420.327/0001-85,LOC BR 230 KM 31,S/N,NaN,ESTRADA,58309602,ETANOL,05/01/2011,"1,79",BRANCA,ca-2011-01.csv,01420327000185,2011-01-05,1.79
4,PB,BAYEUX,ABASTECA COMERCIO DE COMBUSTIVEIS LTDA,07.211.715/0001-32,RUA FRANCISCO MARQUES DA FONSECA,4047,KM 82 MARGENS DA BR 101,ALTO DA BOA VISTA,58308001,GASOLINA,05/01/2011,"2,56",PETROBRAS DISTRIBUIDORA S.A.,ca-2011-01.csv,07211715000132,2011-01-05,2.56


estado                0
municipio             0
revenda               0
cnpj_original         0
logradouro            0
numero                0
complemento       36184
bairro                0
cep                   0
produto               0
data_original         0
valor_original        0
bandeira              0
arquivo_origem        0
cnpj_revenda          0
data                  0
valor                 0
dtype: int64

In [2]:
print("Municípios:", df["municipio"].unique())
print("Produtos:", df["produto"].unique())
print("Período:", df["data"].min(), "até", df["data"].max())

Municípios: <ArrowStringArray>
['BAYEUX', 'CABEDELO', 'SANTA RITA', 'JOAO PESSOA']
Length: 4, dtype: str
Produtos: <ArrowStringArray>
['GASOLINA', 'ETANOL']
Length: 2, dtype: str
Período: 2011-01-04 00:00:00 até 2026-06-29 00:00:00


In [3]:
auditoria = []
rejeitados = []

def aplicar_filtro(base, manter, etapa):
    manter = pd.Series(manter, index=base.index).fillna(False).astype(bool)
    removidos = base.loc[~manter].copy()
    if not removidos.empty:
        removidos["motivo_rejeicao"] = etapa
        rejeitados.append(removidos)
    resultado = base.loc[manter].copy()
    auditoria.append({
        "etapa": etapa,
        "registros_antes": len(base),
        "registros_rejeitados": len(removidos),
        "registros_depois": len(resultado)
    })
    return resultado

for coluna in ["municipio", "produto", "estado", "revenda", "bairro", "logradouro"]:
    if coluna in df.columns:
        df[coluna] = (
            df[coluna].astype("string").str.strip().str.upper()
            .str.replace(r"\s+", " ", regex=True)
        )

df["data"] = pd.to_datetime(df["data"], errors="coerce")
df["valor"] = pd.to_numeric(
    df["valor"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce"
)
df["cnpj_revenda"] = (
    df["cnpj_revenda"].astype("string").str.replace("[^0-9]", "", regex=True).str.zfill(14)
)

def validar_cnpj(cnpj):
    if pd.isna(cnpj) or len(cnpj) != 14 or len(set(cnpj)) == 1:
        return False
    digitos = [int(digito) for digito in cnpj]
    for tamanho, pesos in [
        (12, [5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2]),
        (13, [6, 5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2])
    ]:
        verificador = (sum(a * b for a, b in zip(digitos[:tamanho], pesos)) * 10) % 11
        if verificador == 10:
            verificador = 0
        if verificador != digitos[tamanho]:
            return False
    return True

df["cnpj_valido"] = df["cnpj_revenda"].map(validar_cnpj)
df["id_posto"] = df["cnpj_revenda"].where(df["cnpj_valido"])

df = aplicar_filtro(df, df["produto"].isin(["GASOLINA", "ETANOL"]), "Produto fora do escopo")
df = aplicar_filtro(df, df["data"].notna(), "Data ausente ou inválida")
df = aplicar_filtro(
    df,
    df["valor"].notna() & np.isfinite(df["valor"].fillna(0)) & (df["valor"] > 0),
    "Preço ausente, não numérico ou não positivo"
)

df["semana"] = df["data"].dt.to_period("W-SUN").dt.start_time
df["duplicado_exato"] = df.duplicated(
    subset=["cnpj_revenda", "produto", "data", "valor", "arquivo_origem"],
    keep=False
)
print("Registros com CNPJ inválido preservados:", int((~df["cnpj_valido"]).sum()))
print("Duplicidades exatas sinalizadas e preservadas:", int(df["duplicado_exato"].sum()))

Registros com CNPJ inválido preservados: 0
Duplicidades exatas sinalizadas e preservadas: 0


In [4]:
# Sinaliza candidatos para revisão; nenhum critério estatístico remove preços automaticamente.
grupo_antigo = df.groupby(["produto", "semana"])["valor"]
media_antiga = grupo_antigo.transform("mean")
desvio_antigo = grupo_antigo.transform("std")
contagem_antiga = grupo_antigo.transform("count")
df["outlier_criterio_original_3sd"] = (
    (contagem_antiga >= 3)
    & (desvio_antigo > 0)
    & ((df["valor"] - media_antiga).abs() > 3 * desvio_antigo)
)

grupo_local = df.groupby(["municipio", "produto", "semana"])["valor"]
q1 = grupo_local.transform(lambda valores: valores.quantile(0.25))
q3 = grupo_local.transform(lambda valores: valores.quantile(0.75))
contagem_local = grupo_local.transform("count")
iqr = q3 - q1
df["outlier_iqr_local"] = (
    (contagem_local >= 4)
    & (iqr > 0)
    & ((df["valor"] < q1 - 1.5 * iqr) | (df["valor"] > q3 + 1.5 * iqr))
)
df["outlier_sinalizado"] = df["outlier_criterio_original_3sd"] | df["outlier_iqr_local"]

PASTA_ANALITICO = RAIZ / "Dados/Analitico"
PASTA_ANALITICO.mkdir(parents=True, exist_ok=True)
df.to_parquet(RAIZ / "Dados/Limpo/precos_rmjp_limpos.parquet", index=False)
df.loc[df["outlier_sinalizado"]].to_csv(
    PASTA_TABELAS / "outliers_sinalizados.csv", index=False, sep=";", decimal=","
)
if rejeitados:
    pd.concat(rejeitados, ignore_index=True).to_csv(
        PASTA_TABELAS / "registros_rejeitados.csv", index=False, sep=";", decimal=","
    )
pd.DataFrame(auditoria).to_csv(
    PASTA_TABELAS / "auditoria_limpeza.csv", index=False, sep=";", decimal=","
)

sinalizados_por_ambos = int((df["outlier_criterio_original_3sd"] & df["outlier_iqr_local"]).sum())
print("Registros válidos preservados:", len(df))
print("Candidatos pelo critério original de 3 desvios-padrão:", int(df["outlier_criterio_original_3sd"].sum()))
print("Candidatos pelo IQR local:", int(df["outlier_iqr_local"].sum()))
print("Sinalizados pelos dois critérios:", sinalizados_por_ambos)
print("Outliers sinalizados, sem exclusão automática:", int(df["outlier_sinalizado"].sum()))
print("Percentual sinalizado:", round(100 * df["outlier_sinalizado"].mean(), 2), "%")
print("Registros rejeitados por critérios de validade:", sum(len(bloco) for bloco in rejeitados))
display(pd.DataFrame(auditoria))

Registros válidos preservados: 69506
Candidatos pelo critério original de 3 desvios-padrão: 694
Candidatos pelo IQR local: 5098
Sinalizados pelos dois critérios: 335
Outliers sinalizados, sem exclusão automática: 5457
Percentual sinalizado: 7.85 %
Registros rejeitados por critérios de validade: 0


,etapa,registros_antes,registros_rejeitados,registros_depois
0,Produto fora do escopo,69506,0,69506
1,Data ausente ou inválida,69506,0,69506
2,"Preço ausente, não numérico ou não positivo",69506,0,69506
